# L29 — Factorial Design for Simulation Experiments

**Module**: M09 | **Chapter**: 12 | **Lecture**: L29

## Learning Objectives
By the end of this notebook you will be able to:
1. Construct a 2^k full factorial design for k=2 and k=3 factors.
2. Estimate main effects and two-factor interactions from simulation output.
3. Interpret an interaction plot to identify when effects are not additive.
4. Screen factors using a 2^k-1 fractional factorial design.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

We study a two-stage tandem queue (registration → triage nurse) varying:
- Factor A: Number of registration clerks (1 or 2)
- Factor B: Number of triage nurses (1 or 2)
- Factor C: Patient arrival rate (λ=4/hr or λ=6/hr)
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
from scipy import stats
from itertools import product

## 1. Tandem Queue Simulation

In [ ]:
def run_tandem(lam, mu_reg, mu_nurse, n_clerks, n_nurses,
               n_patients=500, warmup=100, rng=None):
    """
    Two-stage tandem queue: registration → triage nurse.
    Returns mean total sojourn time W (arrival to end of triage).
    """
    if rng is None:
        rng = np.random.default_rng()

    env     = simpy.Environment()
    clerks  = simpy.Resource(env, capacity=n_clerks)
    nurses  = simpy.Resource(env, capacity=n_nurses)
    sojourns = []

    def patient():
        t_arrival = env.now

        # Stage 1: Registration
        with clerks.request() as req:
            yield req
            yield env.timeout(rng.exponential(1.0 / mu_reg))

        # Stage 2: Triage
        with nurses.request() as req:
            yield req
            yield env.timeout(rng.exponential(1.0 / mu_nurse))

        sojourns.append(env.now - t_arrival)

    def arrivals():
        for _ in range(n_patients):
            env.process(patient())
            yield env.timeout(rng.exponential(1.0 / lam))

    env.process(arrivals())
    env.run()
    return np.mean(sojourns[warmup:])


# Quick check
w = run_tandem(lam=5/60, mu_reg=20/60, mu_nurse=7.5/60,
               n_clerks=1, n_nurses=1, rng=np.random.default_rng(42))
print(f"Single run W = {w:.2f} min")

## 2. 2^2 Full Factorial — Two Factors

Factors:
- A: n_clerks ∈ {1, 2}  (coded: −1, +1)
- B: n_nurses ∈ {1, 2}  (coded: −1, +1)

Response: mean sojourn time W.

In [ ]:
N_REPS = 20
LAM    = 5.0 / 60   # 5 patients/hour
MU_REG = 20.0 / 60  # 20 reg/hour
MU_NRS = 7.5 / 60   # 7.5 triage/hour

factor_levels = {
    'A_clerks':  [1, 2],
    'B_nurses':  [1, 2],
}

design_22 = pd.DataFrame(
    list(product(*factor_levels.values())),
    columns=list(factor_levels.keys())
)
print("2^2 design matrix:")
print(design_22.to_string(index=False))

# Run each treatment with N_REPS replications
results_22 = []
for _, row in design_22.iterrows():
    reps = []
    for r in range(N_REPS):
        rng = np.random.default_rng(int(row['A_clerks'] * 100 + row['B_nurses'] * 10 + r))
        w   = run_tandem(LAM, MU_REG, MU_NRS,
                         n_clerks=int(row['A_clerks']),
                         n_nurses=int(row['B_nurses']),
                         rng=rng)
        reps.append(w)
    results_22.append(np.mean(reps))

design_22['W_mean'] = np.round(results_22, 3)
print("\n2^2 results:")
print(design_22.to_string(index=False))

## 3. Main Effects and Interaction Estimates

For a 2^2 design (each factor at −1 / +1):

$$\text{Effect A} = \bar{Y}_{A+} - \bar{Y}_{A-}$$
$$\text{Interaction AB} = \tfrac{1}{2}[(Y_{++} - Y_{-+}) - (Y_{+-} - Y_{--})]$$

In [ ]:
y = design_22.set_index(['A_clerks', 'B_nurses'])['W_mean']

# Treatment means
y11 = y[1, 1]; y12 = y[1, 2]; y21 = y[2, 1]; y22 = y[2, 2]

effect_A  = ((y21 + y22) - (y11 + y12)) / 2
effect_B  = ((y12 + y22) - (y11 + y21)) / 2
effect_AB = ((y22 - y12) - (y21 - y11)) / 2

print(f"Main effect A (clerks):    {effect_A:+.3f} min")
print(f"Main effect B (nurses):    {effect_B:+.3f} min")
print(f"Interaction AB:            {effect_AB:+.3f} min")
print()
print("A negative effect means adding a resource reduces wait.")
print("Interaction: the effect of A depends on the level of B (non-additivity).")

## 4. Interaction Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for n_nurses, col, marker in [(1, 'steelblue', 'o'), (2, 'tomato', 's')]:
    xs = [1, 2]
    ys = [y[nc, n_nurses] for nc in xs]
    ax.plot(xs, ys, color=col, marker=marker, lw=2, ms=8,
            label=f'n_nurses={n_nurses}')

ax.set_xlabel('Number of registration clerks (A)')
ax.set_ylabel('Mean sojourn time W (min)')
ax.set_title('Interaction plot: A × B  (tandem queue)')
ax.set_xticks([1, 2])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Parallel lines → no interaction (effects are additive).")
print("Crossing or converging lines → interaction present.")

## 5. 2^3 Full Factorial — Three Factors

In [ ]:
factor_levels_3 = {
    'A_clerks': [1, 2],
    'B_nurses': [1, 2],
    'C_lam':    [4/60, 6/60],  # 4 or 6 patients/hour
}

design_23 = pd.DataFrame(
    list(product(*factor_levels_3.values())),
    columns=['A_clerks', 'B_nurses', 'C_lam']
)
print(f"2^3 design: {len(design_23)} treatments")

results_23 = []
for _, row in design_23.iterrows():
    reps = []
    for r in range(N_REPS):
        rng = np.random.default_rng(
            int(row['A_clerks'] * 1000 + row['B_nurses'] * 100 + int(row['C_lam']*1000) + r)
        )
        w = run_tandem(row['C_lam'], MU_REG, MU_NRS,
                       n_clerks=int(row['A_clerks']),
                       n_nurses=int(row['B_nurses']),
                       rng=rng)
        reps.append(w)
    results_23.append(np.mean(reps))

design_23['W_mean'] = np.round(results_23, 3)
design_23['C_label'] = design_23['C_lam'].apply(lambda x: f'λ={x*60:.0f}/hr')
print(design_23.drop('C_lam', axis=1).to_string(index=False))

In [ ]:
# Compute all main effects and interactions using Yates algorithm
# Coded: A=-1/+1 for clerks, B=-1/+1 for nurses, C=-1/+1 for lam

def yates_effects(y_ordered):
    """Yates algorithm for 2^3. y_ordered in standard order: (1), a, b, ab, c, ac, bc, abc."""
    n = len(y_ordered)   # 8
    col = list(y_ordered)
    for _ in range(3):   # k=3 factors
        col = [col[i] + col[i+1] if i % 2 == 0 else col[i] - col[i-1]
               for i in range(n)]
    grand_mean = col[0] / n
    effects = [c / (n // 2) for c in col[1:]]
    return grand_mean, effects

# Standard Yates order: (1), a, b, ab, c, ac, bc, abc
# Our design_23 rows: A∈{1,2}, B∈{1,2}, C∈{4,6}
# We map (1)=11-4, a=21-4, b=12-4, ab=22-4, c=11-6, ac=21-6, bc=12-6, abc=22-6
d = design_23.set_index(['A_clerks','B_nurses','C_label'])
order = [(1,'λ=4/hr'), (2,'λ=4/hr'),(1,'λ=4/hr'),(2,'λ=4/hr'),
         (1,'λ=6/hr'),(2,'λ=6/hr'),(1,'λ=6/hr'),(2,'λ=6/hr')]
b_order = [1,1,2,2,1,1,2,2]

y_std = [design_23[
    (design_23['A_clerks']==a) & (design_23['B_nurses']==b) & (design_23['C_label']==c)
]['W_mean'].values[0]
    for (a,c), b in zip(order, b_order)]

gm, effs = yates_effects(y_std)
labels = ['A', 'B', 'AB', 'C', 'AC', 'BC', 'ABC']
print(f"Grand mean: {gm:.3f} min")
for lab, e in zip(labels, effs):
    print(f"  Effect {lab:5s} = {e:+.3f} min")

## 6. Pareto Chart of Effects

In [ ]:
effect_df = pd.DataFrame({'factor': labels, 'effect': effs})
effect_df = effect_df.reindex(effect_df['effect'].abs().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if e < 0 else 'tomato' for e in effect_df['effect']]
ax.barh(effect_df['factor'], effect_df['effect'].abs(), color=colors, alpha=0.8)
ax.set_xlabel('|Effect| on W (min)')
ax.set_title('Pareto chart of 2^3 effects  (blue=negative, red=positive)')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Try It Yourself

1. **Resolution III fractional factorial**: The 2^(3-1) half-fraction uses 4 runs instead of 8. Use the defining relation I=ABC: include only the runs where A·B·C=+1 (i.e., all three at the same level). Estimate the main effects from these 4 runs only. Are the estimates close to the full factorial? Which effects are aliased (confounded) with each other?

2. **Center points**: Add center points to the 2^2 design (A_clerks=1.5, B_nurses=1.5 — round to nearest integer). Run 5 center-point replications. Compute the curvature: if the average center-point response differs significantly from the average of the corner responses, the true response surface is curved, not planar.

3. **Response surface**: Fit a linear regression model W = β_0 + β_A·A + β_B·B + β_AB·A·B to the 2^2 results. What does the model predict for (A=1.5, B=1.5)? Is this close to your simulation estimate at that point?